In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

spark.version  # confirm Spark is running

'3.5.0'

In [2]:
def generate_sk(cols):
    return sha2(concat_ws("||", *cols), 256)

<!-- Dimension: dim_customers -->

In [3]:
customers_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/customers.csv")
orders_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/orders.csv")
order_items_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/order_items.csv")

cust_orders = customers_df.join(orders_df, "customer_id")


w = Window.partitionBy("customer_unique_id").orderBy(col("order_purchase_timestamp").desc())

latest = cust_orders.withColumn("rn", row_number().over(w)) \
    .filter("rn = 1")

agg = cust_orders.groupBy("customer_unique_id").agg(
    min("order_purchase_timestamp").alias("first_order_date"),
    countDistinct("order_id").alias("total_orders")
)

spend = order_items_df.join(orders_df, "order_id") \
    .join(customers_df, "customer_id") \
    .groupBy("customer_unique_id") \
    .agg(sum(col("price") + col("freight_value")).alias("total_spend"))

dim_customers = latest.join(agg, "customer_unique_id") \
    .join(spend, "customer_unique_id", "left") \
    .select(
        generate_sk([col("customer_unique_id")]).alias("customer_key"),
        "customer_unique_id",
        col("customer_city"),
        col("customer_state"),
        col("customer_zip_code_prefix"),
        col("first_order_date").cast("date"),
        "total_orders",
        col("total_spend").cast("decimal(10,2)"),
        (col("total_orders") > 1).alias("is_repeat_customer")
    )

dim_customers.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/gold/dim_customers.csv")

<!-- Dimension: dim_products -->

In [4]:
products_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/products.csv")
dim_products = products_df.select(
    generate_sk([col("product_id")]).alias("product_key"),
    "product_id",
    coalesce(col("product_category_name"), lit("unknown")).alias("product_category_name"),
    col("product_weight_g").cast("double"),
    (
        col("product_length_cm") *
        col("product_height_cm") *
        col("product_width_cm")
    ).alias("product_volume_cm3"),
    "product_photos_qty",
    "product_description_lenght"
)

dim_products.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/gold/dim_products.csv")

<!-- Dimension: dim_sellers -->

In [5]:
sellers_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/sellers.csv")
dim_sellers = sellers_df.select(
    generate_sk([col("seller_id")]).alias("seller_key"),
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix"
)
dim_sellers.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/gold/dim_sellers.csv")

<!-- Fact Table: fact_order_items -->

In [6]:
orders_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/orders.csv")
order_items_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/order_items.csv")
payments_df = spark.read.option("header", True).option("inferSchema",True).csv("/home/jovyan/blue-owls-de-assessment/submission/output/silver/payments.csv")

fact = order_items_df \
    .join(orders_df, "order_id") \
    .join(dim_products, "product_id") \
    .join(dim_sellers, "seller_id") \
    .join(customers_df, "customer_id")

payment_agg = payments_df.groupBy("order_id").agg(sum("payment_value").alias("total_payment"),
                                                  max("payment_installments").alias("payment_installments"))

w = Window.partitionBy("order_id").orderBy(col("payment_value").desc(), col("payment_type"))
payment_type = payments_df.withColumn("rn", row_number().over(w)).filter("rn = 1").select("order_id", "payment_type")

price_sum = order_items_df.groupBy("order_id").agg(sum("price").alias("total_price"))

fact = fact.join(price_sum, "order_id").join(payment_agg, "order_id").join(payment_type, "order_id")

fact = fact.withColumn("payment_value",(col("price") / col("total_price")) * col("total_payment"))
fact = fact.withColumn("days_to_deliver",datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
           .withColumn("days_delivery_vs_estimate",datediff("order_delivered_customer_date", "order_estimated_delivery_date")) \
           .withColumn("is_late_delivery",col("days_delivery_vs_estimate") > 0)

fact_order_items = fact.select(
    generate_sk([col("order_id")]).alias("order_item_sk"),
    "order_id",
    "order_item_id",
    generate_sk([col("customer_unique_id")]).alias("customer_key"),
    "product_key",
    "seller_key",
    col("order_purchase_timestamp").cast("date").alias("order_date"),
    "order_status",
    col("price").cast("double"),
    col("freight_value").cast("double"),
    "payment_value",
    "payment_type",
    "payment_installments",
    "days_to_deliver",
    "days_delivery_vs_estimate",
    "is_late_delivery"
)
fact_order_items.coalesce(1).write.format("csv").mode("overwrite").option("header",True).save("/home/jovyan/blue-owls-de-assessment/submission/output/gold/fact_order_items.csv")



In [83]:
fact_order_items_df = spark.read.option("header", True).option("inferSchema",True).csv("output/gold/fact_order_items.csv")
fact_order_items_df.createOrReplaceTempView("fact_order_items")

dim_products_df = spark.read.option("header", True).option("inferSchema",True).csv("output/gold/dim_products.csv")
dim_products_df.createOrReplaceTempView("dim_products")

In [84]:
df=spark.sql("""WITH base AS (
    SELECT 
        dp.product_category_name,
        YEAR(f.order_date) AS year,
        MONTH(f.order_date) AS month,
        (f.price + f.freight_value) AS revenue
    FROM fact_order_items f
    JOIN dim_products dp
        ON f.product_key = dp.product_key
),

-- Monthly aggregation
monthly AS (
    SELECT
        product_category_name,
        year,
        month,
        SUM(revenue) AS monthly_revenue,
        COUNT(*) AS txn_count
    FROM base
    GROUP BY product_category_name, year, month
),

-- Filter categories with at least 10 transactions
filtered AS (
    SELECT *
    FROM monthly
    WHERE txn_count >= 10
),

-- Rank categories within each month
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY year, month
            ORDER BY monthly_revenue DESC
        ) AS monthly_rank
    FROM filtered
),

-- Top 5 categories by overall revenue
top_categories AS (
    SELECT product_category_name
    FROM ranked
    GROUP BY product_category_name
    ORDER BY SUM(monthly_revenue) DESC
    LIMIT 5
),

-- Keep only top categories
final_base AS (
    SELECT r.*
    FROM ranked r
    JOIN top_categories t
        ON r.product_category_name = t.product_category_name
),

-- Add MoM and rolling avg
final AS (
    SELECT
        product_category_name,
        year,
        month,
        monthly_revenue,
        monthly_rank,

        -- MoM Growth %
        ROUND(
            (
                (monthly_revenue - LAG(monthly_revenue) OVER (
                    PARTITION BY product_category_name
                    ORDER BY year, month
                ))
                /
                LAG(monthly_revenue) OVER (
                    PARTITION BY product_category_name
                    ORDER BY year, month
                )
            ) * 100,
            2
        ) AS mom_growth_pct,

        -- Rolling 3-month average
        ROUND(
            AVG(monthly_revenue) OVER (
                PARTITION BY product_category_name
                ORDER BY year, month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ),
            2
        ) AS rolling_3m_avg_revenue

    FROM final_base
)

SELECT *
FROM final
ORDER BY product_category_name, year, month;""")

In [87]:
df.show()

+---------------------+----+-----+------------------+------------+--------------+----------------------+
|product_category_name|year|month|   monthly_revenue|monthly_rank|mom_growth_pct|rolling_3m_avg_revenue|
+---------------------+----+-----+------------------+------------+--------------+----------------------+
|         beleza_saude|2018|    7|          97886.33|           2|          NULL|              97886.33|
|         beleza_saude|2018|    8|117619.04000000014|           1|         20.16|             107752.69|
|      cama_mesa_banho|2018|    7| 52592.61000000004|           4|          NULL|              52592.61|
|      cama_mesa_banho|2018|    8| 59356.29999999996|           4|         12.86|              55974.46|
|        esporte_lazer|2018|    7| 51820.11000000003|           5|          NULL|              51820.11|
|        esporte_lazer|2018|    8| 50372.91000000003|           5|         -2.79|              51096.51|
|   relogios_presentes|2018|    7| 98107.12000000018|  